# On-Policy Distillation on SageMaker HyperPod EKS

Compress a frontier **teacher** (`Qwen/Qwen3-4B`) into a lean **student** (`Qwen/Qwen3-0.6B`)
by training on the student's *own* math rollouts with **dense, per-token feedback** (reverse KL
to the teacher). No verifier — cheaper and more stable than RL.

**Everything runs on the cluster** `hp-cluster-onpolicy-distillation` (ns `default`). This notebook is just
the **control panel**: it deploys the teacher, launches the student training Job, and monitors.
Nothing trains in the notebook kernel — so a CPU kernel is fine.

**Flow:** 0 Setup → 1 Deploy teacher → 2 Launch student Job → 3 Monitor convergence → 4 Cleanup.

## Architecture — two GPU pools, decoupled

Teacher = long-lived vLLM inference `Service` (static weights, scoring only). Student = separate
training `Job`. Each pod gets a **dedicated GPU** (`nvidia.com/gpu: 1`) — never shared. The only
coupling is the cluster network (the `Service`) and the shared FSx/S3 mounts.

```mermaid
flowchart LR
  subgraph TP["TEACHER POOL — dedicated GPU · inference only"]
    T["Deployment teacher-vllm<br/>vllm serve Qwen3-4B<br/>static weights"]
    SVC(["Service teacher-vllm:8000"])
    T --> SVC
  end
  subgraph SP["STUDENT POOL — dedicated GPU · training"]
    S["Job opd-student<br/>train_distill.py<br/>generate rollouts + update"]
  end
  subgraph STO["SHARED STORAGE"]
    FSX[("FSx /fsx<br/>HF cache · checkpoints · metrics")]
    S3[("S3 /s3<br/>models · code")]
  end
  S -- "1 . student token_ids" --> SVC
  SVC -- "2 . per-token teacher logprobs" --> S
  S -. read/write .-> FSX
  S -. read .-> S3
  T -. read .-> S3
```

**Per-step loop:** ① student samples a rollout → ② teacher scores every token → advantage =
−(log π_student − log π_teacher) → student updates. Repeat until the teacher-KL stops dropping.

## Part 0 — Setup

Download and install kubectl

In [ ]:
%%bash --out bash_env

HP_CLUSTER_NAME=chakra-test
EKS_CLUSTER_ARN=`aws sagemaker describe-cluster --cluster-name $HP_CLUSTER_NAME --query "Orchestrator.Eks.ClusterArn" --output text`
EKS_CLUSTER_NAME=`echo $EKS_CLUSTER_ARN|awk -F / '{print $2}'`
K8S_VERSION=`aws eks describe-cluster --name $EKS_CLUSTER_NAME --query "cluster.version" --output text`
S3_URL="https://docs.aws.amazon.com/eks/latest/userguide/install-kubectl.html#linux_amd64_kubectl"
KUBECTL_PATH=$(curl -s "${S3_URL}/"| grep -Po 'https://[^<]*'"${K8S_VERSION}"'[^<]*linux/amd64/kubectl()?') 
KUBECTL_PATH=`echo ${KUBECTL_PATH}| cut -d ' ' -f 1`
#Download and install kubectl
curl -O $KUBECTL_PATH
mkdir -p ~/bin
chmod +x kubectl
mv kubectl ~/bin
echo $EKS_CLUSTER_ARN

Point `kubectl` at the cluster and define a small `sh()` shell helper. (`kubectl` lives in `~/bin`,
so we add it to PATH and call it by absolute path.)

In [ ]:
!pip install -q matplotlib 2>/dev/null

In [ ]:
import os, json, subprocess
os.environ["PATH"] = os.path.expanduser("~/bin") + os.pathsep + "/usr/local/bin" + os.pathsep + os.environ.get("PATH", "")

KUBECTL   = os.path.expanduser("~/bin/kubectl")
CTX       = bash_env.strip()
NAMESPACE = "default"
KCTL      = f"{KUBECTL} --context {CTX} -n {NAMESPACE}"
RUN_DIR   = "/fsx/opd/run1"          # where the Job writes checkpoints + metrics (on FSx)


def sh(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    print(r.stdout or r.stderr); return r
def sh_out(cmd):
    return subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True).stdout

sh("which kubectl aws envsubst")                              # sanity: tools resolve
sh(f"{KCTL} get nodes -L node.kubernetes.io/instance-type")  # the GPU pools
sh(f"{KCTL} get pods -o wide")                               # what's already running
 

#### Get the S3 Bucket name attached as PV(s3-claim) and EKS CLuster ARN and update env_vars

In [ ]:
#EKS Cluster ARN
print(f"EKS Cluster ARN: {bash_env.strip()}")

In [ ]:
#S3 Bucket - s3-claim
%%bash
kubectl get pv $(kubectl get pvc s3-claim -o jsonpath='{.spec.volumeName}') -o jsonpath='{"Bucket: "}{.spec.csi.volumeAttributes.bucketName}{"\n"}'

**Note:** 
1. Copy the EKS_CLUSTER_ARN from above and update "export CTX=" in manifests/env_vars
2. Copy the Bucket from above and update "S3_BUCKET" in manifests/env_vars

## Part 1 — Deploy the teacher (vLLM inference Service)

A vLLM `Deployment` + `Service` serving `Qwen/Qwen3-4B`, reading weights from the S3 mount and
caching to FSx. It exposes an OpenAI-compatible API and returns per-token log-probs (echo-logprobs)
— that's the only thing the student needs from it. Model + storage + instance type come from
`manifests/env_vars`.

> First deploy pulls a ~10 GB image + the model, so `rollout status` can take several minutes.

In [ ]:
sh("set -a && source manifests/env_vars && set +a && "
   f"envsubst < manifests/teacher-vllm.yaml-template | {KUBECTL} --context {CTX} apply -f -")
sh(f"{KCTL} rollout status deploy/teacher-vllm --timeout=1200s")   # returns once /v1/models is ready
sh(f"{KCTL} get pods -l app=teacher-vllm -o wide")

## Part 2 — Launch the student (distillation Job)

Sync the trainer code to S3, then submit the student `Job`. The Job waits for the teacher `Service`,
then runs `src/train_distill.py`: it generates rollouts, asks the teacher to score them, computes the
per-token reverse-KL advantage, and updates the student — all on its own GPU, inside the cluster.

In [ ]:
# 1) make the code available to the pod at $CODE_DIR (/s3/opd/src)
sh("set -a && source manifests/env_vars && set +a && aws s3 sync src s3://$S3_BUCKET/opd/src")
# 2) create the training Job
sh("set -a && source manifests/env_vars && set +a && "
   f"envsubst < manifests/student-distill-job.yaml-template | {KUBECTL} --context {CTX} apply -f -")
# 3) both pools now on the cluster: teacher-vllm (inference) + opd-student (training)
sh(f"{KCTL} get pods -o wide")

## Part 3 — Monitor convergence

Watch the Job logs — `teacher_kl` should trend **down** as the student matches the teacher. The plot
reads the metrics file the Job writes on FSx (via the teacher pod, which shares the `/fsx` mount).
Re-run these cells to refresh while training runs.

In [ ]:
sh(f"{KCTL} get pods -o wide")
sh(f"{KCTL} logs job/opd-student --tail=40")     # per-step teacher_kl; re-run to refresh

In [ ]:
import matplotlib.pyplot as plt
tpod = sh_out(f"{KCTL} get pods -l app=teacher-vllm -o jsonpath='{{.items[0].metadata.name}}'").strip()
raw  = sh_out(f"{KCTL} exec {tpod} -c vllm -- cat {RUN_DIR}/metrics_rank0.jsonl")
rows = [json.loads(l) for l in raw.splitlines() if l.strip()]
if rows:
    plt.figure(figsize=(7, 3.5))
    plt.plot([r["step"] for r in rows], [r["teacher_kl"] for r in rows], marker="o")
    plt.xlabel("step"); plt.ylabel("mean teacher reverse KL")
    plt.title("Convergence: student -> teacher"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f"{len(rows)} steps · latest teacher_kl = {rows[-1]['teacher_kl']:.4f}")
else:
    print("No metrics yet — the Job is still starting (image pull + teacher wait). Re-run shortly.")

## Part 4 — Cleanup

Delete the training Job and the teacher Service to free the GPUs.

In [ ]:
sh(f"{KCTL} delete job opd-student --ignore-not-found")
sh(f"{KCTL} delete deploy,svc teacher-vllm --ignore-not-found")